[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/09-system-design/05_consistency.ipynb)

*System Design · Notebook 5 of 10*

# Consistency & Distributed Systems

What happens when machines disagree: CAP and PACELC, Lamport and vector clocks, clock skew, last-writer-wins, CRDTs, idempotency, two-phase commit vs sagas, majorities and split brain, Raft voting, leases and fencing tokens, heartbeats, gossip and Merkle-tree anti-entropy.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Distributed systems are colleagues in different cities with no shared clock: they must agree on what happened, in what order, even when calls drop.

**What interviewers really test here:** CAP and PACELC, logical and vector clocks, idempotency, 2PC vs sagas, quorums and Raft, leases, fencing and gossip.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. CAP, PACELC and consistency models *(Basics)*
- [ ] 2. Time and ordering *(Core)*
- [ ] 3. Idempotency and distributed transactions *(Core)*
- [ ] 4. Consensus, leaders and failure detection *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import json
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Fallacies of distributed computing

- The network is reliable — it isn't; retry with idempotency.
- Latency is zero — every hop costs; batch and cache.
- Bandwidth is infinite; the network is secure; topology doesn't change.
- There is one administrator; transport cost is zero; the network is homogeneous.

### 2PC vs saga

|   | Two-phase commit | Saga |
| --- | --- | --- |
| Atomicity | true atomic commit | eventual, via compensations |
| Locks | held across services | local only |
| Failure | blocks if coordinator dies | keeps going, must design undo steps |
| Use | within one database cluster | across microservices |

---
# Part 2: Lessons, step by step

## Lesson 1: CAP, PACELC and consistency models

*Level: Basics*

Networks fail. When a **partition** splits your nodes, each side must either refuse requests (stay **consistent**, CP) or keep answering with possibly stale data (stay **available**, AP). **PACELC** adds: even without partitions you trade **latency** against consistency, because waiting for replicas takes time.



| Model | Promise |
| --- | --- |
| Linearizable (strong) | behaves like one copy; every read sees the latest write |
| Read-your-writes | you always see your own updates |
| Monotonic reads | you never see time go backwards |
| Eventual | replicas converge if writes stop |

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 2: Time and ordering

*Level: Core*

Machine clocks drift and jump (NTP corrections), so wall-clock timestamps can order events wrongly. **Lamport clocks** give an order consistent with cause and effect; **vector clocks** also detect when two updates were **concurrent** (a true conflict). Conflicts are resolved by last-writer-wins (simple, loses data), by the application (merge carts), or with **CRDTs** — data types whose merges always converge.

**✏️ Your turn:** you now have what you need for [Q10](#q10), [Q12](#q12), [Q11](#q11), [Q13](#q13), [Q2](#q2), [Q3](#q3). Try one now, or carry on and do them in Part 3.

## Lesson 3: Idempotency and distributed transactions

*Level: Core*

Over a network you can't tell "request lost" from "response lost", so clients retry — and every handler that changes state must be **idempotent**: store a request id and return the saved result on repeats. True exactly-once delivery doesn't exist; at-least-once delivery plus idempotent processing gives **effectively once**.



For a change spanning services: **two-phase commit** (prepare, then commit) is atomic but blocks if the coordinator dies. A **saga** runs local transactions and on failure runs compensating actions (refund, release stock) — the usual choice for microservices.

**✏️ Your turn:** you now have what you need for [Q14](#q14), [Q8](#q8), [Q15](#q15), [Q16](#q16). Try one now, or carry on and do them in Part 3.

## Lesson 4: Consensus, leaders and failure detection

*Level: Advanced*

Many systems need one leader (one writer, one scheduler). **Raft** and Paxos elect it with a **majority** vote: a 5-node cluster survives 2 failures, and a minority side of a partition can't elect a second leader — no split brain. Raft voters only accept candidates whose log is at least as complete as theirs.




- **Heartbeats** detect failure, but slow and dead look the same — so timeouts are a trade-off.
- A **lease** lets a leader act for a fixed time; **fencing tokens** stop an old leader that wakes up late.
- **Gossip** spreads membership and state in O(log n) rounds; **Merkle trees** find which data differs between replicas cheaply.



In practice use etcd, ZooKeeper or Consul rather than writing consensus yourself.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def lamport_demo():
    clock = {"A": 0, "B": 0}
    clock["A"] += 1; sent = clock["A"]; print("A sends at", sent)
    clock["B"] += 1; print("B local event at", clock["B"])
    clock["B"] = max(clock["B"], sent) + 1; print("B receives at", clock["B"], "— always after the send")


lamport_demo()

A sends at 1
B local event at 1
B receives at 2 — always after the send


**✏️ Your turn:** you now have what you need for [Q4](#q4), [Q17](#q17), [Q9](#q9), [Q5](#q5), [Q18](#q18), [Q6](#q6), [Q7](#q7), [Q19](#q19). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [3]:
from collections import defaultdict


<a id="q1"></a>
### Q1. CP or AP?

🟢 Warm-up · *What to give up during a partition*

For each system, return 'CP' (refuse requests rather than return wrong data) if a stale or conflicting answer is harmful, else 'AP' (stay available, reconcile later). Each item is `(name, stale_is_harmful)`.

**Examples**

- `systems = [('bank balance', True), ('social likes', False), ('seat booking', True), ('shopping cart', False), ('DNS', False), ('distributed lock', True)]` → `[('bank balance', 'CP'), ('social likes', 'AP'), ('seat booking', 'CP'), ('shopping cart', 'AP'), ('DNS', 'AP'), ('distributed lock', 'CP')]`

In [ ]:
def cap_choice(systems):
    """
    For each system, return 'CP' (refuse requests rather than return wrong data)
    if a stale or conflicting answer is harmful, else 'AP' (stay available,
    reconcile later). Each item is (name, stale_is_harmful).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(cap_choice, [
    (([('bank balance', True), ('social likes', False), ('seat booking', True), ('shopping cart', False), ('DNS', False), ('distributed lock', True)],), [('bank balance', 'CP'), ('social likes', 'AP'), ('seat booking', 'CP'), ('shopping cart', 'AP'), ('DNS', 'AP'), ('distributed lock', 'CP')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

If/else per system.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** If/else per system.

```python
def cap_choice(systems):
    out = []
    for name, harmful in systems:
        if harmful:
            out.append((name, "CP"))
        else:
            out.append((name, "AP"))
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension.

```python
def cap_choice(systems):
    return [(name, "CP" if harmful else "AP") for name, harmful in systems]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: If/else per system. That costs O(n) time and O(n) space."

"I can do better: A comprehension. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/consistency/cp-or-ap)

<a id="q2"></a>
### Q2. Last Writer Wins

🟢 Warm-up · *Simple conflict resolution*

Concurrent writes `(value, timestamp, node)` to the same key are resolved by the highest timestamp, ties broken by node id. Return the surviving value and the values silently lost.

**Examples**

- `writes = [('blue', 1005, 'n2'), ('green', 1005, 'n3'), ('red', 1001, 'n1')]` → `('green', ['blue', 'red'])`

In [ ]:
def lww(writes):
    """
    Concurrent writes (value, timestamp, node) to the same key are resolved by
    the highest timestamp, ties broken by node id. Return the surviving value
    and the values silently lost.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(lww, [
    (([('blue', 1005, 'n2'), ('green', 1005, 'n3'), ('red', 1001, 'n1')],), ('green', ['blue', 'red'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Find the max with a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`max` on (timestamp, node).

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Find the max with a loop.

```python
def lww(writes):
    best = writes[0]
    for w in writes[1:]:
        if (w[1], w[2]) > (best[1], best[2]):
            best = w
    lost = []
    for w in writes:
        if w is not best:
            lost.append(w[0])
    return (best[0], sorted(lost))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `max` on (timestamp, node).

```python
def lww(writes):
    win = max(writes, key=lambda w: (w[1], w[2]))
    return (win[0], sorted(w[0] for w in writes if w is not win))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Find the max with a loop. That costs O(n) time and O(n) space."

"I can do better: `max` on (timestamp, node). That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/consistency/last-writer-wins)

<a id="q3"></a>
### Q3. Merge a Grow-Only Counter (CRDT)

🟢 Warm-up · *Conflict-free by design*

Each replica keeps a count per node. Merging takes the max per node; the counter's value is the sum. Return the merged state (sorted pairs) and the value. Merging in any order gives the same result.

**Examples**

- `replicas = [{'a': 5, 'b': 2}, {'a': 3, 'b': 4, 'c': 1}, {'c': 2}]` → `([('a', 5), ('b', 4), ('c', 2)], 11)`

In [ ]:
def gcounter(replicas):
    """
    Each replica keeps a count per node. Merging takes the max per node; the
    counter's value is the sum. Return the merged state (sorted pairs) and the
    value. Merging in any order gives the same result.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(gcounter, [
    (([{'a': 5, 'b': 2}, {'a': 3, 'b': 4, 'c': 1}, {'c': 2}],), ([('a', 5), ('b', 4), ('c', 2)], 11)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over replicas and nodes.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Dict comprehension over the union of nodes.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop over replicas and nodes.

```python
def gcounter(replicas):
    merged = {}
    for r in replicas:
        for node, c in r.items():
            if c > merged.get(node, 0):
                merged[node] = c
    return (sorted(merged.items()), sum(merged.values()))
```

**Cost:** O(r · n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Dict comprehension over the union of nodes.

```python
def gcounter(replicas):
    nodes = set().union(*replicas)
    merged = {n: max(r.get(n, 0) for r in replicas) for n in nodes}
    return (sorted(merged.items()), sum(merged.values()))
```

**Cost:** O(r · n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over replicas and nodes. That costs O(r · n) time and O(n) space."

"I can do better: Dict comprehension over the union of nodes. That's O(r · n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/consistency/merge-a-grow-only-counter-crdt)

<a id="q4"></a>
### Q4. Which Side Keeps Working After a Split?

🟢 Warm-up · *Majority quorums*

A cluster of `n` nodes splits into groups of the given sizes. Only a group with a strict majority may elect a leader and accept writes (this prevents split brain). Return a bool per group and the failures an n-node cluster tolerates.

**Examples**

- `n = 5, groups = [3, 2]` → `([True, False], 2)`
- `n = 4, groups = [2, 2]` → `([False, False], 1)`
- `n = 7, groups = [3, 2, 2]` → `([False, False, False], 3)`

In [ ]:
def majority(n, groups):
    """
    A cluster of n nodes splits into groups of the given sizes. Only a group
    with a strict majority may elect a leader and accept writes (this prevents
    split brain). Return a bool per group and the failures an n-node cluster
    tolerates.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(majority, [
    ((5, [3, 2]), ([True, False], 2)),
    ((4, [2, 2]), ([False, False], 1)),
    ((7, [3, 2, 2]), ([False, False, False], 3)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare each size with n/2.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Comprehension with `n // 2 + 1`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Compare each size with n/2.

```python
def majority(n, groups):
    out = []
    for g in groups:
        out.append(g > n / 2)
    return (out, (n - 1) // 2)
```

**Cost:** O(g) time, O(g) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Comprehension with `n // 2 + 1`.

```python
def majority(n, groups):
    return ([g >= n // 2 + 1 for g in groups], (n - 1) // 2)
```

**Cost:** O(g) time, O(g) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare each size with n/2. That costs O(g) time and O(g) space."

"I can do better: Comprehension with `n // 2 + 1`. That's O(g) time and O(g) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/consistency/which-side-keeps-working-after-a-split)

<a id="q5"></a>
### Q5. When Did the Leader Lease Lapse?

🟢 Warm-up · *Leases*

A leader renews a lease of `lease` seconds at each heartbeat time. Return the `(start, end)` gaps where the lease had expired before the next renewal (no one may act as leader then).

**Examples**

- `beats = [0, 4, 8, 20, 23, 35], lease = 5` → `[(13, 20), (28, 35)]`

In [ ]:
def lease_gaps(beats, lease):
    """
    A leader renews a lease of lease seconds at each heartbeat time. Return the
    (start, end) gaps where the lease had expired before the next renewal (no
    one may act as leader then).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(lease_gaps, [
    (([0, 4, 8, 20, 23, 35], 5), [(13, 20), (28, 35)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare consecutive heartbeats.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`zip` over neighbours.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Compare consecutive heartbeats.

```python
def lease_gaps(beats, lease):
    gaps = []
    for i in range(1, len(beats)):
        if beats[i] - beats[i - 1] > lease:
            gaps.append((beats[i - 1] + lease, beats[i]))
    return gaps
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `zip` over neighbours.

```python
def lease_gaps(beats, lease):
    return [(a + lease, b) for a, b in zip(beats, beats[1:]) if b - a > lease]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare consecutive heartbeats. That costs O(n) time and O(n) space."

"I can do better: `zip` over neighbours. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/consistency/when-did-the-leader-lease-lapse)

<a id="q6"></a>
### Q6. A Heartbeat Failure Detector

🟢 Warm-up · *Suspect silent nodes*

Given each node's last heartbeat time, the current time and a timeout, return the nodes to suspect (sorted). Note a slow node and a dead node look the same.

**Examples**

- `last_beat = {'n1': 98, 'n2': 91, 'n3': 100, 'n4': 70}, now = 100, timeout = 5` → `['n2', 'n4']`

In [ ]:
def suspects(last_beat, now, timeout):
    """
    Given each node's last heartbeat time, the current time and a timeout,
    return the nodes to suspect (sorted). Note a slow node and a dead node look
    the same.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(suspects, [
    (({'n1': 98, 'n2': 91, 'n3': 100, 'n4': 70}, 100, 5), ['n2', 'n4']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop and compare.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sorted comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop and compare.

```python
def suspects(last_beat, now, timeout):
    out = []
    for node, t in last_beat.items():
        if now - t > timeout:
            out.append(node)
    out.sort()
    return out
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Sorted comprehension.

```python
def suspects(last_beat, now, timeout):
    return sorted(n for n, t in last_beat.items() if now - t > timeout)
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop and compare. That costs O(n log n) time and O(n) space."

"I can do better: Sorted comprehension. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/consistency/a-heartbeat-failure-detector)

<a id="q7"></a>
### Q7. How Fast Does Gossip Spread?

🟢 Warm-up · *Epidemic protocols*

One node knows an update. Each round every informed node tells `fanout` new nodes. Return the rounds needed to reach all `n` nodes for each n — it grows like log n.

**Examples**

- `ns = [10, 100, 1000, 1000000], fanout = 2` → `[3, 5, 7, 13]`

In [ ]:
def gossip_rounds(ns, fanout):
    """
    One node knows an update. Each round every informed node tells fanout new
    nodes. Return the rounds needed to reach all n nodes for each n — it grows
    like log n.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(gossip_rounds, [
    (([10, 100, 1000, 1000000], 2), [3, 5, 7, 13]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Simulate round by round.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Multiply by (1 + fanout) each round.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Simulate round by round.

```python
def gossip_rounds(ns, fanout):
    out = []
    for n in ns:
        informed, rounds = 1, 0
        while informed < n:
            informed = informed + informed * fanout
            rounds += 1
        out.append(rounds)
    return out
```

**Cost:** O(log n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Multiply by (1 + fanout) each round.

```python
def gossip_rounds(ns, fanout):
    def rounds(n):
        r, reach = 0, 1
        while reach < n:
            r, reach = r + 1, reach * (1 + fanout)
        return r
    return [rounds(n) for n in ns]
```

**Cost:** O(log n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Simulate round by round. That costs O(log n) time and O(1) space."

"I can do better: Multiply by (1 + fanout) each round. That's O(log n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/consistency/how-fast-does-gossip-spread)

<a id="q8"></a>
### Q8. Deduplicate by Sequence Number

🟢 Warm-up · *Effectively-once delivery*

Producers number their messages. The consumer remembers the highest sequence per producer and drops anything not higher (a retry or replay). Return `(accepted count, dropped messages)`.

**Examples**

- `messages = [('p1', 1), ('p1', 2), ('p2', 1), ('p1', 2), ('p1', 3), ('p2', 1), ('p2', 2), ('p1', 1)]` → `(5, [('p1', 2), ('p2', 1), ('p1', 1)])`

In [ ]:
def seq_dedupe(messages):
    """
    Producers number their messages. The consumer remembers the highest sequence
    per producer and drops anything not higher (a retry or replay). Return
    (accepted count, dropped messages).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(seq_dedupe, [
    (([('p1', 1), ('p1', 2), ('p2', 1), ('p1', 2), ('p1', 3), ('p2', 1), ('p2', 2), ('p1', 1)],), (5, [('p1', 2), ('p2', 1), ('p1', 1)])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Dict of last sequence numbers.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same, using `dict.get`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Dict of last sequence numbers.

```python
def seq_dedupe(messages):
    last, accepted, dropped = {}, 0, []
    for p, s in messages:
        if p in last and s <= last[p]:
            dropped.append((p, s))
        else:
            last[p] = s
            accepted += 1
    return (accepted, dropped)
```

**Cost:** O(n) time, O(p) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same, using `dict.get`.

```python
def seq_dedupe(messages):
    last, dropped = {}, []
    for p, s in messages:
        if s > last.get(p, 0):
            last[p] = s
        else:
            dropped.append((p, s))
    return (len(messages) - len(dropped), dropped)
```

**Cost:** O(n) time, O(p) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Dict of last sequence numbers. That costs O(n) time and O(p) space."

"I can do better: Same, using `dict.get`. That's O(n) time and O(p) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/consistency/deduplicate-by-sequence-number)

<a id="q9"></a>
### Q9. How Many Nodes for Consensus?

🟢 Warm-up · *Crash vs Byzantine faults*

To survive f crashed nodes, Raft/Paxos needs 2f + 1 nodes (majority f + 1). To survive f nodes that lie (Byzantine), BFT protocols need 3f + 1. Return `(raft nodes, raft majority, BFT nodes)` for each f.

**Examples**

- `fs = [1, 2, 3]` → `[(3, 2, 4), (5, 3, 7), (7, 4, 10)]`

In [ ]:
def consensus_size(fs):
    """
    To survive f crashed nodes, Raft/Paxos needs 2f + 1 nodes (majority f + 1).
    To survive f nodes that lie (Byzantine), BFT protocols need 3f + 1. Return
    (raft nodes, raft majority, BFT nodes) for each f.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(consensus_size, [
    (([1, 2, 3],), [(3, 2, 4), (5, 3, 7), (7, 4, 10)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A loop.

```python
def consensus_size(fs):
    out = []
    for f in fs:
        out.append((2 * f + 1, f + 1, 3 * f + 1))
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension.

```python
def consensus_size(fs):
    return [(2 * f + 1, (2 * f + 1) // 2 + 1, 3 * f + 1) for f in fs]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(n) time and O(n) space."

"I can do better: A comprehension. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/consistency/how-many-nodes-for-consensus)

<a id="q10"></a>
### Q10. Lamport Clocks

🟡 Interview standard · *Logical time*

Events run in this global order: `(process, 'local'|'send'|'recv', message)`. Apply Lamport's rules and return each event's timestamp.

**Examples**

- `events = [('A', 'local', None), ('A', 'send', 'm1'), ('B', 'local', None), ('B', 'recv', 'm1'), ('B', 'send', 'm2'), ('A', 'local', None), ('C', 'recv', 'm2'), ('A', 'recv', 'm2')]` → `[1, 2, 1, 3, 4, 3, 5, 5]`

In [ ]:
def lamport(events):
    """
    Events run in this global order: (process, 'local'|'send'|'recv', message).
    Apply Lamport's rules and return each event's timestamp.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(lamport, [
    (([('A', 'local', None), ('A', 'send', 'm1'), ('B', 'local', None), ('B', 'recv', 'm1'), ('B', 'send', 'm2'), ('A', 'local', None), ('C', 'recv', 'm2'), ('A', 'recv', 'm2')],), [1, 2, 1, 3, 4, 3, 5, 5]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Clock dict per process and if/elif.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same, compacted with `max`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Clock dict per process and if/elif.

```python
def lamport(events):
    clock, sent, out = {}, {}, []
    for p, kind, msg in events:
        c = clock.get(p, 0)
        if kind == "recv":
            c = max(c, sent[msg]) + 1
        else:
            c = c + 1
        if kind == "send":
            sent[msg] = c
        clock[p] = c
        out.append(c)
    return out
```

**Cost:** O(e) time, O(p + m) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same, compacted with `max`.

```python
def lamport(events):
    clock, sent, out = defaultdict(int), {}, []
    for p, kind, msg in events:
        clock[p] = max(clock[p], sent[msg] if kind == "recv" else 0) + 1
        if kind == "send":
            sent[msg] = clock[p]
        out.append(clock[p])
    return out
```

**Cost:** O(e) time, O(p + m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Clock dict per process and if/elif. That costs O(e) time and O(p + m) space."

"I can do better: Same, compacted with `max`. That's O(e) time and O(p + m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/consistency/lamport-clocks)

<a id="q11"></a>
### Q11. Compare Vector Clocks

🟡 Interview standard · *Happened-before or concurrent?*

For each pair of vector clocks (dicts node → counter, missing = 0) return 'before', 'after', 'equal' or 'concurrent'. Concurrent versions are real conflicts the application must merge.

**Examples**

- `pairs = [({'A': 1, 'B': 2}, {'A': 2, 'B': 2}), ({'A': 3}, {'A': 2, 'B': 1}), ({'A': 1, 'B': 1}, {'A': 1, 'B': 1}), ({'A': 2, 'B': 0}, {'A': 1})]` → `['before', 'concurrent', 'equal', 'after']`

In [ ]:
def compare_vc(pairs):
    """
    For each pair of vector clocks (dicts node → counter, missing = 0) return
    'before', 'after', 'equal' or 'concurrent'. Concurrent versions are real
    conflicts the application must merge.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(compare_vc, [
    (([({'A': 1, 'B': 2}, {'A': 2, 'B': 2}), ({'A': 3}, {'A': 2, 'B': 1}), ({'A': 1, 'B': 1}, {'A': 1, 'B': 1}), ({'A': 2, 'B': 0}, {'A': 1})],), ['before', 'concurrent', 'equal', 'after']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Track 'some less' and 'some greater' flags.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`all` over the union of keys.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Track 'some less' and 'some greater' flags.

```python
def compare_vc(pairs):
    out = []
    for a, b in pairs:
        less = greater = False
        for k in set(a) | set(b):
            x, y = a.get(k, 0), b.get(k, 0)
            if x < y:
                less = True
            if x > y:
                greater = True
        if less and greater:
            out.append("concurrent")
        elif less:
            out.append("before")
        elif greater:
            out.append("after")
        else:
            out.append("equal")
    return out
```

**Cost:** O(p · k) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `all` over the union of keys.

```python
def compare_vc(pairs):
    def cmp(a, b):
        keys = a.keys() | b.keys()
        le = all(a.get(k, 0) <= b.get(k, 0) for k in keys)
        ge = all(a.get(k, 0) >= b.get(k, 0) for k in keys)
        return "equal" if le and ge else "before" if le else "after" if ge else "concurrent"
    return [cmp(a, b) for a, b in pairs]
```

**Cost:** O(p · k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Track 'some less' and 'some greater' flags. That costs O(p · k) time and O(k) space."

"I can do better: `all` over the union of keys. That's O(p · k) time and O(k) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `pairs = [({'A': 1}, {'A': 2})]`

1. GeneratorExit raised: 
2. cmp() returns 'before'
3. compare\_vc() returns ['before']

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/consistency/compare-vector-clocks)

<a id="q12"></a>
### Q12. Run Vector Clocks

🟡 Interview standard · *One counter per process*

Using the same event format as the Lamport question, keep a vector clock per process (on receive, take the element-wise max, then increment your own entry). Return each process's final clock as a sorted list of `(node, counter)` pairs.

**Examples**

- `events = [('A', 'local', None), ('A', 'send', 'm1'), ('B', 'recv', 'm1'), ('B', 'send', 'm2'), ('C', 'local', None), ('C', 'recv', 'm2')]` → `{'A': [('A', 2)], 'B': [('A', 2), ('B', 2)], 'C': [('A', 2), ('B', 2), ('C', 2)]}`

In [ ]:
def vector_clocks(events):
    """
    Using the same event format as the Lamport question, keep a vector clock per
    process (on receive, take the element-wise max, then increment your own
    entry). Return each process's final clock as a sorted list of (node,
    counter) pairs.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(vector_clocks, [
    (([('A', 'local', None), ('A', 'send', 'm1'), ('B', 'recv', 'm1'), ('B', 'send', 'm2'), ('C', 'local', None), ('C', 'recv', 'm2')],), {'A': [('A', 2)], 'B': [('A', 2), ('B', 2)], 'C': [('A', 2), ('B', 2), ('C', 2)]}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Dicts of dicts with explicit loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`defaultdict(Counter)` and dict merges.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Dicts of dicts with explicit loops.

```python
def vector_clocks(events):
    clocks, sent = {}, {}
    for p, kind, msg in events:
        if p not in clocks:
            clocks[p] = {}
        if kind == "recv":
            for k, v in sent[msg].items():
                if v > clocks[p].get(k, 0):
                    clocks[p][k] = v
        clocks[p][p] = clocks[p].get(p, 0) + 1
        if kind == "send":
            sent[msg] = dict(clocks[p])
    result = {}
    for p in clocks:
        result[p] = sorted(clocks[p].items())
    return result
```

**Cost:** O(e · p) time, O(p²) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `defaultdict(Counter)` and dict merges.

```python
def vector_clocks(events):
    clocks, sent = defaultdict(dict), {}
    for p, kind, msg in events:
        c = clocks[p]
        if kind == "recv":
            c.update({k: max(v, c.get(k, 0)) for k, v in sent[msg].items()})
        c[p] = c.get(p, 0) + 1
        if kind == "send":
            sent[msg] = dict(c)
    return {p: sorted(c.items()) for p, c in clocks.items()}
```

**Cost:** O(e · p) time, O(p²) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Dicts of dicts with explicit loops. That costs O(e · p) time and O(p²) space."

"I can do better: `defaultdict(Counter)` and dict merges. That's O(e · p) time and O(p²) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/consistency/run-vector-clocks)

<a id="q13"></a>
### Q13. Clock Skew Reorders Events

🟡 Interview standard · *Why wall clocks lie*

Events `(node, true_ms, name)` are stamped with the node's wall clock = true time + `skew[node]`. Return the order a system sorting by wall clock would see, and how many pairs of events end up in the wrong order.

**Examples**

- `events = [('n1', 100, 'create'), ('n2', 105, 'update'), ('n1', 110, 'delete'), ('n3', 120, 'restore')], skew = {'n1': 0, 'n2': -20, 'n3': -15}` → `(['update', 'create', 'restore', 'delete'], 2)`

In [ ]:
def skew_order(events, skew):
    """
    Events (node, true_ms, name) are stamped with the node's wall clock = true
    time + skew[node]. Return the order a system sorting by wall clock would
    see, and how many pairs of events end up in the wrong order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(skew_order, [
    (([('n1', 100, 'create'), ('n2', 105, 'update'), ('n1', 110, 'delete'), ('n3', 120, 'restore')], {'n1': 0, 'n2': -20, 'n3': -15}), (['update', 'create', 'restore', 'delete'], 2)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort, then compare every pair.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same pair count with a comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Sort, then compare every pair.

```python
def skew_order(events, skew):
    stamped = sorted(events, key=lambda e: e[1] + skew[e[0]])
    order = [e[2] for e in stamped]
    wrong = 0
    for i in range(len(stamped)):
        for j in range(i + 1, len(stamped)):
            if stamped[i][1] > stamped[j][1]:
                wrong += 1
    return (order, wrong)
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same pair count with a comprehension.

```python
def skew_order(events, skew):
    s = sorted(events, key=lambda e: e[1] + skew[e[0]])
    return ([e[2] for e in s], sum(s[i][1] > s[j][1] for i in range(len(s)) for j in range(i + 1, len(s))))
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort, then compare every pair. That costs O(n²) time and O(n) space."

"I can do better: Same pair count with a comprehension. That's O(n²) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/consistency/clock-skew-reorders-events)

<a id="q14"></a>
### Q14. Make the Payment Idempotent

🟡 Interview standard · *Retries without double charges*

Clients retry on timeouts, so the same request can arrive twice. Apply each `(request_id, amount)` debit once; repeats return the stored result. Return `(final balance, duplicates ignored)`; compare with applying everything.

**Examples**

- `balance = 1000, requests = [('r1', 200), ('r2', 50), ('r1', 200), ('r3', 300), ('r2', 50), ('r1', 200)]` → `(450, 3)`

In [ ]:
def idempotent(balance, requests):
    """
    Clients retry on timeouts, so the same request can arrive twice. Apply each
    (request_id, amount) debit once; repeats return the stored result. Return
    (final balance, duplicates ignored); compare with applying everything.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(idempotent, [
    ((1000, [('r1', 200), ('r2', 50), ('r1', 200), ('r3', 300), ('r2', 50), ('r1', 200)]), (450, 3)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A list of seen ids.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A set of seen ids.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A list of seen ids.

```python
def idempotent(balance, requests):
    seen, dupes = [], 0
    for rid, amount in requests:
        if rid in seen:
            dupes += 1
        else:
            seen.append(rid)
            balance -= amount
    return (balance, dupes)
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A set of seen ids.

```python
def idempotent(balance, requests):
    done = {}
    for rid, amount in requests:
        done.setdefault(rid, amount)
    return (balance - sum(done.values()), len(requests) - len(done))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A list of seen ids. That costs O(n²) time and O(n) space."

"I can do better: A set of seen ids. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/consistency/make-the-payment-idempotent)

<a id="q15"></a>
### Q15. Two-Phase Commit

🟡 Interview standard · *All or nothing across services*

For each scenario `(votes, coordinator_alive_after_prepare)`: if any participant votes 'no', 'abort'; if all vote 'yes' but the coordinator crashes before sending the decision, participants are stuck 'blocked' holding locks; otherwise 'commit'.

**Examples**

- `scenarios = [(['yes', 'yes', 'yes'], True), (['yes', 'no', 'yes'], True), (['yes', 'yes'], False), (['no', 'yes'], False)]` → `['commit', 'abort', 'blocked', 'abort']`

In [ ]:
def two_pc(scenarios):
    """
    For each scenario (votes, coordinator_alive_after_prepare): if any
    participant votes 'no', 'abort'; if all vote 'yes' but the coordinator
    crashes before sending the decision, participants are stuck 'blocked'
    holding locks; otherwise 'commit'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(two_pc, [
    (([(['yes', 'yes', 'yes'], True), (['yes', 'no', 'yes'], True), (['yes', 'yes'], False), (['no', 'yes'], False)],), ['commit', 'abort', 'blocked', 'abort']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

If/elif per scenario.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A conditional expression.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** If/elif per scenario.

```python
def two_pc(scenarios):
    out = []
    for votes, alive in scenarios:
        if "no" in votes:
            out.append("abort")
        elif not alive:
            out.append("blocked")
        else:
            out.append("commit")
    return out
```

**Cost:** O(s · p) time, O(s) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A conditional expression.

```python
def two_pc(scenarios):
    return ["abort" if "no" in v else "commit" if alive else "blocked" for v, alive in scenarios]
```

**Cost:** O(s · p) time, O(s) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: If/elif per scenario. That costs O(s · p) time and O(s) space."

"I can do better: A conditional expression. That's O(s · p) time and O(s) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/consistency/two-phase-commit)

<a id="q16"></a>
### Q16. Run a Saga

🟡 Interview standard · *Compensate instead of rolling back*

A saga runs local steps in order. If a step fails, run the compensations of the completed steps in reverse. Steps are `(name, succeeds)`; return the actions executed (compensation of X is 'undo X').

**Examples**

- `steps = [('reserve stock', True), ('charge card', True), ('book courier', False), ('send email', True)]` → `['reserve stock', 'charge card', 'book courier', 'undo charge card', 'undo reserve stock']`
- `steps = [('reserve stock', True), ('charge card', True)]` → `['reserve stock', 'charge card']`

In [ ]:
def saga(steps):
    """
    A saga runs local steps in order. If a step fails, run the compensations of
    the completed steps in reverse. Steps are (name, succeeds); return the
    actions executed (compensation of X is 'undo X').
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(saga, [
    (([('reserve stock', True), ('charge card', True), ('book courier', False), ('send email', True)],), ['reserve stock', 'charge card', 'book courier', 'undo charge card', 'undo reserve stock']),
    (([('reserve stock', True), ('charge card', True)],), ['reserve stock', 'charge card']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop, and on failure loop back over done steps.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same with `reversed`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop, and on failure loop back over done steps.

```python
def saga(steps):
    actions, done = [], []
    for name, ok in steps:
        actions.append(name)
        if not ok:
            i = len(done) - 1
            while i >= 0:
                actions.append("undo " + done[i])
                i -= 1
            return actions
        done.append(name)
    return actions
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same with `reversed`.

```python
def saga(steps):
    actions = []
    for i, (name, ok) in enumerate(steps):
        actions.append(name)
        if not ok:
            return actions + [f"undo {n}" for n, _ in reversed(steps[:i])]
    return actions
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop, and on failure loop back over done steps. That costs O(n) time and O(n) space."

"I can do better: Same with `reversed`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/consistency/run-a-saga)

<a id="q17"></a>
### Q17. Who Wins the Raft Vote?

🟡 Interview standard · *Up-to-date logs only*

A candidate with log end `(last_term, last_index)` asks the other nodes for votes. A node votes yes only if the candidate's log is at least as up-to-date as its own (compare term, then index). The candidate votes for itself. Return `(votes, wins a majority?)` for each candidate.

**Examples**

- `candidates = [(5, 10), (5, 8), (4, 20)], voters = [(5, 9), (5, 10), (4, 30), (3, 2)]` → `[(5, True), (3, True), (2, False)]`

In [ ]:
def raft_vote(candidates, voters):
    """
    A candidate with log end (last_term, last_index) asks the other nodes for
    votes. A node votes yes only if the candidate's log is at least as up-to-
    date as its own (compare term, then index). The candidate votes for itself.
    Return (votes, wins a majority?) for each candidate.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(raft_vote, [
    (([(5, 10), (5, 8), (4, 20)], [(5, 9), (5, 10), (4, 30), (3, 2)]), [(5, True), (3, True), (2, False)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count with explicit comparisons.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Tuple comparison inside `sum`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Count with explicit comparisons.

```python
def raft_vote(candidates, voters):
    n = len(voters) + 1
    out = []
    for ct, ci in candidates:
        votes = 1
        for vt, vi in voters:
            if ct > vt or (ct == vt and ci >= vi):
                votes += 1
        out.append((votes, votes > n // 2))
    return out
```

**Cost:** O(c · n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Tuple comparison inside `sum`.

```python
def raft_vote(candidates, voters):
    need = (len(voters) + 1) // 2 + 1
    out = []
    for cand in candidates:
        votes = 1 + sum(cand >= v for v in voters)
        out.append((votes, votes >= need))
    return out
```

**Cost:** O(c · n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count with explicit comparisons. That costs O(c · n) time and O(1) space."

"I can do better: Tuple comparison inside `sum`. That's O(c · n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/consistency/who-wins-the-raft-vote)

<a id="q18"></a>
### Q18. Fencing Tokens

🟡 Interview standard · *Stop the zombie lock holder*

A lock service hands out increasing tokens. The storage remembers the highest token seen and rejects writes with a lower one — so a client that paused (GC, network) after its lock expired can't corrupt data. Writes are `(client, token)`; return the accepted clients in order.

**Examples**

- `writes = [('A', 33), ('B', 34), ('A', 33), ('B', 34), ('C', 35), ('B', 34)]` → `['A', 'B', 'B', 'C']`

In [ ]:
def fencing(writes):
    """
    A lock service hands out increasing tokens. The storage remembers the
    highest token seen and rejects writes with a lower one — so a client that
    paused (GC, network) after its lock expired can't corrupt data. Writes are
    (client, token); return the accepted clients in order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(fencing, [
    (([('A', 33), ('B', 34), ('A', 33), ('B', 34), ('C', 35), ('B', 34)],), ['A', 'B', 'B', 'C']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Track the max token in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same, one condition.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Track the max token in a loop.

```python
def fencing(writes):
    highest, accepted = -1, []
    for client, token in writes:
        if token >= highest:
            highest = token
            accepted.append(client)
    return accepted
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same, one condition.

```python
def fencing(writes):
    highest, accepted = -1, []
    for client, token in writes:
        if token >= highest:
            highest, accepted = token, accepted + [client]
    return accepted
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Track the max token in a loop. That costs O(n) time and O(1) space."

"I can do better: Same, one condition. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/consistency/fencing-tokens)

<a id="q19"></a>
### Q19. Find Divergent Buckets (Merkle Anti-Entropy)

🟡 Interview standard · *Compare hashes, not data*

Two replicas split keys into `b` buckets by `crc32(key) % b`. Instead of sending all data, they compare one hash per bucket and only sync buckets whose hashes differ. Return the differing buckets.

**Examples**

- `a = {'k1': 1, 'k2': 2, 'k3': 3, 'k4': 4, 'k5': 5}, b = {'k1': 1, 'k2': 20, 'k3': 3, 'k4': 4, 'k6': 6}, buckets = 4` → `[0, 2, 3]`

In [ ]:
import hashlib

import zlib

def merkle_diff(a, b, buckets):
    """
    Two replicas split keys into b buckets by crc32(key) % b. Instead of sending
    all data, they compare one hash per bucket and only sync buckets whose
    hashes differ. Return the differing buckets.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(merkle_diff, [
    (({'k1': 1, 'k2': 2, 'k3': 3, 'k4': 4, 'k5': 5}, {'k1': 1, 'k2': 20, 'k3': 3, 'k4': 4, 'k6': 6}, 4), [0, 2, 3]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare the sorted items of each bucket directly.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Hash each bucket with SHA-256 and compare digests.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Compare the sorted items of each bucket directly.

```python
def merkle_diff(a, b, buckets):
    def items(d, i):
        return sorted((k, v) for k, v in d.items() if zlib.crc32(k.encode()) % buckets == i)
    return [i for i in range(buckets) if items(a, i) != items(b, i)]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Hash each bucket with SHA-256 and compare digests.

```python
def merkle_diff(a, b, buckets):
    def digests(d):
        groups = defaultdict(list)
        for k, v in sorted(d.items()):
            groups[zlib.crc32(k.encode()) % buckets].append(f"{k}={v}")
        return [hashlib.sha256("|".join(groups[i]).encode()).hexdigest() for i in range(buckets)]
    da, db = digests(a), digests(b)
    return [i for i in range(buckets) if da[i] != db[i]]
```

**Cost:** O(n) time, O(b) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare the sorted items of each bucket directly. That costs O(n) time and O(n) space."

"I can do better: Hash each bucket with SHA-256 and compare digests. That's O(n) time and O(b) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `a = {'a': 1}, b = {'a': 2}, buckets = 2`

1. digests() returns ['e3b0c44298fc1c149afbf4c8996fb92427a...
2. digests() returns ['e3b0c44298fc1c149afbf4c8996fb92427a...
3. merkle\_diff() returns [1]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/consistency/find-divergent-buckets-merkle-anti-entropy)

<a id="q20"></a>
### Q20. Monotonic Reads

🟡 Interview standard · *Never go back in time*

A client reads from the replicas listed in `reads`; replicas hold different versions. To avoid seeing older data after newer, if a replica is behind the highest version the client has seen, it reads from the primary instead. Return `(server used, version seen)` for each read.

**Examples**

- `replicas = {'r1': 7, 'r2': 5, 'r3': 8}, primary_version = 9, reads = ['r2', 'r1', 'r2', 'r3', 'r1']` → `[('r2', 5), ('r1', 7), ('primary', 9), ('primary', 9), ('primary', 9)]`

In [ ]:
def monotonic_reads(replicas, primary_version, reads):
    """
    A client reads from the replicas listed in reads; replicas hold different
    versions. To avoid seeing older data after newer, if a replica is behind the
    highest version the client has seen, it reads from the primary instead.
    Return (server used, version seen) for each read.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(monotonic_reads, [
    (({'r1': 7, 'r2': 5, 'r3': 8}, 9, ['r2', 'r1', 'r2', 'r3', 'r1']), [('r2', 5), ('r1', 7), ('primary', 9), ('primary', 9), ('primary', 9)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Track the max version seen in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same with a conditional expression.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Track the max version seen in a loop.

```python
def monotonic_reads(replicas, primary_version, reads):
    seen, out = 0, []
    for r in reads:
        if replicas[r] >= seen:
            out.append((r, replicas[r]))
            seen = replicas[r]
        else:
            out.append(("primary", primary_version))
            seen = primary_version
    return out
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same with a conditional expression.

```python
def monotonic_reads(replicas, primary_version, reads):
    seen, out = 0, []
    for r in reads:
        server, v = (r, replicas[r]) if replicas[r] >= seen else ("primary", primary_version)
        out.append((server, v))
        seen = v
    return out
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Track the max version seen in a loop. That costs O(n) time and O(1) space."

"I can do better: Same with a conditional expression. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/consistency/monotonic-reads)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Explain CAP in one minute.</b></summary>

When a network partition happens, a distributed system must choose between answering every request (availability) and never returning stale data (consistency). Without partitions you can have both, but you still trade latency for consistency (PACELC).

</details>

<details>
<summary><b>2. Why are clusters usually 3 or 5 nodes?</b></summary>

Consensus needs a majority. 3 nodes tolerate 1 failure, 5 tolerate 2; an even count adds a node without adding tolerance (4 still tolerates only 1).

</details>

<details>
<summary><b>3. How do you make an API idempotent?</b></summary>

Have clients send an idempotency key; store the key with the result in the same transaction as the change; on a repeat, return the stored result instead of doing the work again.

</details>

<details>
<summary><b>4. What is split brain?</b></summary>

Two nodes both believing they are leader after a partition, both accepting writes. Prevent it with majority quorums and fencing tokens.

</details>

<details>
<summary><b>5. Is exactly-once delivery possible?</b></summary>

Not over an unreliable network. Systems achieve exactly-once *effects* with at-least-once delivery plus deduplication or idempotent writes (and transactions, as in Kafka).

</details>

<details>
<summary><b>6. When would you use a CRDT?</b></summary>

For data edited concurrently in many places without coordination — counters, sets, collaborative text, offline-first apps — where every replica must converge automatically.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: CAP, PACELC and consistency models
- [ ] I can explain: Time and ordering
- [ ] I can explain: Idempotency and distributed transactions
- [ ] I can explain: Consensus, leaders and failure detection

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Run a Saga
- [ ] [Q17](#q17) Who Wins the Raft Vote?
- [ ] [Q18](#q18) Fencing Tokens
- [ ] [Q19](#q19) Find Divergent Buckets (Merkle Anti-Entropy)
- [ ] [Q20](#q20) Monotonic Reads

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.